In [ ]:
# Wine Quality Prediction using Decision Tree

## Machine Learning Group Assignment  
### Individual Contribution - Decision Tree Model

This notebook presents my individual contribution to the supervised learning assignment.  
The task is to predict wine quality using physicochemical properties of wine samples and the wine type.  
A Decision Tree model is used as the selected algorithm for this part of the project.

In [ ]:
## 1. Problem Statement

The aim of this task is to build a supervised machine learning model that can predict the quality of wine using its measured chemical properties.  
This is a multiclass classification problem because the target variable, `quality`, contains several discrete score values.  
The selected model for this individual contribution is a Decision Tree Classifier.

In [ ]:
## 2. Dataset Description

The dataset used in this notebook is the `winequalityN` dataset.  
It contains physicochemical features such as acidity, sugar, chlorides, sulfur dioxide values, density, pH, sulphates, alcohol, and wine type.  
The target variable is `quality`, which represents the wine quality score.

This notebook uses the uploaded dataset file:
- `winequalityN.csv`

Expected input features:
- type
- fixed acidity
- volatile acidity
- citric acid
- residual sugar
- chlorides
- free sulfur dioxide
- total sulfur dioxide
- density
- pH
- sulphates
- alcohol

Target:
- quality

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# Configuration

DATA_PATH = "winequalityN.csv"
MODEL_OUTPUT = "wine_quality_pipeline.pkl"
TARGET_COLUMN = "quality"

FEATURE_COLUMNS = [
    "type",
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]

TEST_SIZE = 0.20
RANDOM_STATE = 42
MAX_DEPTH = None

In [ ]:
# Load the dataset

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

In [ ]:
# Basic dataset inspection

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nTarget class distribution:")
print(df[TARGET_COLUMN].value_counts().sort_index())

In [ ]:
## 3. Preprocessing

The preprocessing steps used in this notebook are:

1. Validate that the required feature columns and target column exist.
2. Separate input features and output target.
3. Handle missing numeric values using median imputation.
4. Handle missing categorical values using most frequent imputation.
5. Encode the categorical `type` column using one-hot encoding.
6. Keep preprocessing and model together inside a single pipeline to avoid feature order problems during prediction.

In [ ]:
# Validate required columns

missing_feature_columns = [col for col in FEATURE_COLUMNS if col not in df.columns]
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' is missing from the dataset.")

if missing_feature_columns:
    raise ValueError(f"Missing required feature columns: {missing_feature_columns}")

print("All required columns are present.")

In [ ]:
# Separate features and target

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

if y.isna().any():
    raise ValueError("The target column contains missing values. Please clean the target before training.")

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Build preprocessing pipeline

categorical_columns = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_columns = [col for col in X_train.columns if col not in categorical_columns]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

model = DecisionTreeClassifier(
    random_state=RANDOM_STATE,
    max_depth=MAX_DEPTH
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

print("Pipeline created successfully.")

In [ ]:
## 4. Model Training

A Decision Tree Classifier is used in this notebook.  
This algorithm is suitable because it is easy to understand, easy to explain during viva, supports multiclass classification, and can capture non-linear decision boundaries.

In [ ]:
# Train the model

pipeline.fit(X_train, y_train)
print("Model training completed.")

In [ ]:
# Generate predictions

y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

print("Evaluation Results")
print("------------------")
print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"Macro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
## 5. Saving the Trained Pipeline

The trained preprocessing steps and the Decision Tree model are saved together in one `.pkl` file.  
This is important because the same preprocessing must be applied again during prediction.

In [ ]:
bundle = {
    "pipeline": pipeline,
    "feature_columns": FEATURE_COLUMNS,
    "target_column": TARGET_COLUMN,
    "task_type": "multiclass_classification",
    "model_name": "DecisionTreeClassifier",
    "metrics": {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
    },
    "source_file": DATA_PATH,
}

joblib.dump(bundle, MODEL_OUTPUT)
print(f"Saved trained pipeline bundle to: {MODEL_OUTPUT}")

In [ ]:
## 6. Sample Prediction

The following cell loads the saved pipeline and predicts the wine quality for one sample input.  
This demonstrates how the trained model can later be connected to a frontend or API.

In [ ]:
# Load saved model bundle

saved_bundle = joblib.load(MODEL_OUTPUT)
saved_pipeline = saved_bundle["pipeline"]
expected_features = saved_bundle["feature_columns"]

print("Saved model loaded successfully.")
print("Expected features:", expected_features)

In [ ]:
# Sample input for prediction

sample = {
    "type": "white",
    "fixed acidity": 6.6,
    "volatile acidity": 0.16,
    "citric acid": 0.4,
    "residual sugar": 1.5,
    "chlorides": 0.044,
    "free sulfur dioxide": 48.0,
    "total sulfur dioxide": 143.0,
    "density": 0.9912,
    "pH": 3.54,
    "sulphates": 0.52,
    "alcohol": 12.4
}

sample_df = pd.DataFrame([[sample[col] for col in expected_features]], columns=expected_features)
sample_df

In [ ]:
# Predict quality for the sample

predicted_quality = saved_pipeline.predict(sample_df)[0]
print("Predicted quality:", predicted_quality)

if hasattr(saved_pipeline, "predict_proba"):
    probabilities = saved_pipeline.predict_proba(sample_df)[0]
    class_labels = saved_pipeline.named_steps["model"].classes_
    probability_dict = {str(label): round(float(prob), 4) for label, prob in zip(class_labels, probabilities)}
    print("Class probabilities:")
    print(json.dumps(probability_dict, indent=2))

In [ ]:
## 7. Discussion

The model is able to predict wine quality using the selected physicochemical features.  
However, the dataset is imbalanced, which can make minority classes harder to predict accurately.  
A single Decision Tree may also overfit the training data if it is not controlled properly.

Possible future improvements:
- hyperparameter tuning
- cross-validation
- handling class imbalance more carefully
- comparing with stronger supervised learning algorithms

In [ ]:
## 8. Individual Contribution

My individual contribution to the group assignment is the implementation and evaluation of the Decision Tree model for wine quality prediction.  
This includes dataset validation, preprocessing, model training, evaluation, saving the trained pipeline, and demonstrating sample prediction.

In [ ]:
## 9. Note for Full Group Submission

This notebook covers only one supervised learning algorithm: Decision Tree.  
For the final group submission, three additional supervised learning algorithms should also be implemented and compared using suitable evaluation metrics.

Recommended algorithms for teammates:
- Logistic Regression
- Random Forest
- K-Nearest Neighbors or Support Vector Machine

In [ ]:
## 10. Conclusion

This notebook demonstrated a complete supervised machine learning workflow for predicting wine quality using a Decision Tree model.  
The workflow included loading the dataset, preprocessing, training, evaluation, saving the model, and performing sample prediction.  
This notebook is suitable for GitHub submission and can also support report writing and viva preparation.